<a href="https://www.kaggle.com/code/haquee/titanic-problem-submission-by-sam?scriptVersionId=310330836" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV

In [3]:
test_df = pd.read_csv("/kaggle/input/titanic/test.csv")
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
submission_df = pd.read_csv("/kaggle/input/titanic/test.csv")

In [4]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [6]:
train_df.describe(include= ["O"])

,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Dooley, Mr. Patrick",male,347082,G6,S
freq,1,577,7,4,644


In [7]:
train_df.groupby(["Pclass"], as_index=False)["Survived"].mean()

,Pclass,Survived
0,1,0.629630
1,2,0.472826
2,3,0.242363


In [8]:
train_df.groupby(["Sex"], as_index=False)["Survived"].mean()

,Sex,Survived
0,female,0.742038
1,male,0.188908


In [9]:
train_df.groupby(["Age"], as_index=False)["Survived"].mean()

,Age,Survived
0,0.42,1.0
1,0.67,1.0
2,0.75,1.0
3,0.83,1.0
4,0.92,1.0
...,...,...
83,70.00,0.0
84,70.50,0.0
85,71.00,0.0
86,74.00,0.0


In [10]:
train_df["Age"].isnull().sum()

np.int64(177)

In [11]:
train_df["Age"].fillna(train_df["Age"].mean(), inplace= True)
test_df["Age"].fillna(test_df["Age"].mean(), inplace= True)

/tmp/ipykernel_17/2999709696.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df["Age"].fillna(train_df["Age"].mean(), inplace= True)
/tmp/ipykernel_17/2999709696.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tru

In [12]:
train_df["Age"].describe()

count    891.000000
mean      29.699118
std       13.002015
min        0.420000
25%       22.000000
50%       29.699118
75%       35.000000
max       80.000000
Name: Age, dtype: float64

In [13]:
# labelling the age column based on the specific age group they fall under
train_df['Age'] = pd.cut(
    train_df['Age'],
    bins=[0, 1, 5, 12, 18, 30, 50, 80],
    labels=[i for i in range(7)]
)
test_df['Age'] = pd.cut(
    test_df['Age'],
    bins=[0, 1, 5, 12, 18, 30, 50, 80],
    labels=[i for i in range(7)]
)

In [14]:
train_df["Family_Size"] = train_df["SibSp"] + train_df["Parch"] + 1
test_df["Family_Size"] = test_df["SibSp"] + test_df["Parch"] + 1
train_df.groupby(train_df["Family_Size"], as_index=False)["Survived"].mean()

,Family_Size,Survived
0,1,0.303538
1,2,0.552795
2,3,0.578431
3,4,0.724138
4,5,0.200000
5,6,0.136364
6,7,0.333333
7,8,0.000000
8,11,0.000000


In [15]:
train_df['TicketNumber'] = train_df['Ticket'].apply(lambda x: pd.Series({'Ticket': x.split()[-1]}))
test_df['TicketNumber'] = test_df['Ticket'].apply(lambda x: pd.Series({'Ticket': x.split()[-1]}))
train_df['TicketNumber']

0        21171
1        17599
2      3101282
3       113803
4       373450
        ...   
886     211536
887     112053
888       6607
889     111369
890     370376
Name: TicketNumber, Length: 891, dtype: object

In [16]:
train_df['TicketLocation'] = np.where(train_df['Ticket'].str.split(pat=" ", expand=True)[1].notna(), train_df['Ticket'].str.split(pat=" ", expand=True)[0].apply(lambda x: x.strip()), 'Blank')
test_df['TicketLocation'] = np.where(test_df['Ticket'].str.split(pat=" ", expand=True)[1].notna(), test_df['Ticket'].str.split(pat=" ", expand=True)[0].apply(lambda x: x.strip()), 'Blank')
train_df['TicketLocation'].value_counts()

TicketLocation
Blank         665
PC             60
C.A.           27
STON/O         12
A/5            10
W./C.           9
CA.             8
SOTON/O.Q.      8
A/5.            7
SOTON/OQ        7
STON/O2.        6
CA              6
C               5
S.O.C.          5
SC/PARIS        5
F.C.C.          5
SC/Paris        4
A/4.            3
PP              3
A/4             3
S.O./P.P.       3
SC/AH           3
A./5.           2
P/PP            2
A.5.            2
WE/P            2
SOTON/O2        2
S.C./PARIS      2
S.C./A.4.       1
Fa              1
S.O.P.          1
SO/C            1
S.P.            1
A4.             1
W.E.P.          1
A/S             1
SC              1
SW/PP           1
SCO/W           1
W/C             1
S.W./PP         1
F.C.            1
C.A./SOTON      1
Name: count, dtype: int64

In [17]:
# Checking if having sco, ston, or STON in the ticket prefix means that someone embarked from SouthHampton
mask_soton_ticket = train_df['TicketLocation'].str.contains(
    r'SOTON|STON|^S\.O|^SO/',
    case=False,
    na=False
)
mask_embarked_s = train_df['Embarked'] == 'S'
# We calculate what percent of ppl with those ppl actually departed from SouthHampton
percent_soton_s = (
    train_df[mask_soton_ticket & mask_embarked_s].shape[0]
    / train_df[mask_soton_ticket].shape[0]
) * 100
percent_soton_s
# As result is 100, we can safely assume that these prefixes mean that a person got on from SouthHampton. 

100.0

In [18]:
# We group ppl into ticket groups based on the ticket prefixes
def map_ticket_group(x):
    if pd.isna(x) or x == 'Blank':
        return 'Blank'
    
    x = x.upper()

    if 'SOTON' in x or 'STON' in x or x.startswith('S.O') or x.startswith('SO/'):
        return 'Southampton'
    
    if 'PARIS' in x or x.startswith('SC'):
        return 'Paris'
    
    if x.startswith('A/') or x.startswith('A.') or x.startswith('W') or 'WE' in x:
        return 'BritishAgency'
    
    if x.startswith('C'):
        return 'Canada'
    
    if x.startswith('PC') or x.startswith('F.C'):
        return 'Premium'
    
    if 'PP' in x or x.startswith('P/'):
        return 'Pooled'
    
    return 'Other'


train_df['TicketGroup'] = train_df['TicketLocation'].apply(map_ticket_group)
test_df['TicketGroup'] = test_df['TicketLocation'].apply(map_ticket_group)

In [19]:
train_df.groupby(train_df["TicketGroup"], as_index= False)[["Survived", "Pclass"]].mean()

,TicketGroup,Survived,Pclass
0,Blank,0.383459,2.351880
1,BritishAgency,0.097561,2.780488
2,Canada,0.347826,2.586957
3,Other,0.000000,3.000000
4,Paris,0.500000,2.000000
5,Pooled,0.714286,2.428571
6,Premium,0.651515,1.075758
7,Southampton,0.239130,2.782609


In [20]:
train_df['Title'] = train_df['Name'].str.split(pat= ",", expand=True)[1].str.split(pat= ".", expand=True)[0].apply(lambda x: x.strip())
test_df['Title'] = test_df['Name'].str.split(pat= ",", expand=True)[1].str.split(pat= ".", expand=True)[0].apply(lambda x: x.strip())

In [21]:
train_df['Title'] = train_df['Title'].replace({
    'Capt': 'Military',
    'Col': 'Military',
    'Major': 'Military',
    'Jonkheer': 'Noble',
    'the Countess': 'Noble',
    'Don': 'Noble',
    'Lady': 'Noble',
    'Sir': 'Noble',
    'Mlle': 'Noble',
    'Ms': 'Noble',
    'Mme': 'Noble'    
})

test_df['Title'] = test_df['Title'].replace({
    'Capt': 'Military',
    'Col': 'Military',
    'Major': 'Military',
    'Jonkheer': 'Noble',
    'the Countess': 'Noble',
    'Don': 'Noble',
    'Lady': 'Noble',
    'Sir': 'Noble',
    'Mlle': 'Noble',
    'Ms': 'Noble',
    'Mme': 'Noble'    
})

In [22]:
train_df['Name_Length'] = train_df['Name'].apply(lambda x: len(x))
test_df['Name_Length'] = test_df['Name'].apply(lambda x: len(x))

In [23]:
train_df['Name_LengthGB'] = pd.qcut(train_df['Name_Length'], 3)
test_df['Name_LengthGB'] = pd.qcut(test_df['Name_Length'], 3)
train_df['Name_LengthGB']

0        (22.0, 28.0]
1        (28.0, 82.0]
2      (11.999, 22.0]
3        (28.0, 82.0]
4        (22.0, 28.0]
            ...      
886    (11.999, 22.0]
887      (22.0, 28.0]
888      (28.0, 82.0]
889    (11.999, 22.0]
890    (11.999, 22.0]
Name: Name_LengthGB, Length: 891, dtype: category
Categories (3, interval[float64, right]): [(11.999, 22.0] < (22.0, 28.0] < (28.0, 82.0]]

In [24]:
train_df.loc[train_df['Name_Length'] <= 22, 'Name_Size'] = 0
train_df.loc[(train_df['Name_Length'] > 22) & (train_df['Name_Length'] <= 28), 'Name_Size'] = 1
train_df.loc[(train_df['Name_Length'] > 28) & (train_df['Name_Length'] <= 82), 'Name_Size'] = 2
train_df.loc[train_df['Name_Length'] > 82, 'Name_Size'] 

test_df.loc[test_df['Name_Length'] <= 22, 'Name_Size'] = 0
test_df.loc[(test_df['Name_Length'] > 22) & (test_df['Name_Length'] <= 28), 'Name_Size'] = 1
test_df.loc[(test_df['Name_Length'] > 28) & (test_df['Name_Length'] <= 82), 'Name_Size'] = 2
test_df.loc[test_df['Name_Length'] > 82, 'Name_Size'] 
train_df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,TicketNumber,TicketLocation,TicketGroup,Title,Name_Length,Name_LengthGB,Name_Size
0,1,0,3,"Braund, Mr. Owen Harris",male,4,1,0,A/5 21171,7.2500,NaN,S,2,21171,A/5,BritishAgency,Mr,23,"(22.0, 28.0]",1.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,5,1,0,PC 17599,71.2833,C85,C,2,17599,PC,Premium,Mrs,51,"(28.0, 82.0]",2.0
2,3,1,3,"Heikkinen, Miss. Laina",female,4,0,0,STON/O2. 3101282,7.9250,NaN,S,1,3101282,STON/O2.,Southampton,Miss,22,"(11.999, 22.0]",0.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,5,1,0,113803,53.1000,C123,S,2,113803,Blank,Blank,Mrs,44,"(28.0, 82.0]",2.0
4,5,0,3,"Allen, Mr. William Henry",male,5,0,0,373450,8.0500,NaN,S,1,373450,Blank,Blank,Mr,24,"(22.0, 28.0]",1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,4,0,0,211536,13.0000,NaN,S,1,211536,Blank,Blank,Rev,21,"(11.999, 22.0]",0.0
887,888,1,1,"Graham, Miss. Margaret Edith",female,4,0,0,112053,30.0000,B42,S,1,112053,Blank,Blank,Miss,28,"(22.0, 28.0]",1.0
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,4,1,2,W./C. 6607,23.4500,NaN,S,4,6607,W./C.,BritishAgency,Miss,40,"(28.0, 82.0]",2.0
889,890,1,1,"Behr, Mr. Karl Howell",male,4,0,0,111369,30.0000,C148,C,1,111369,Blank,Blank,Mr,21,"(11.999, 22.0]",0.0


In [25]:
# we can drop number of parents and siblings as we already calculate family size
train_df.drop(columns=["SibSp", "Parch"], inplace=True)
test_df.drop(columns=["SibSp", "Parch"], inplace=True)

In [26]:
train_df

,PassengerId,Survived,Pclass,Name,Sex,Age,Ticket,Fare,Cabin,Embarked,Family_Size,TicketNumber,TicketLocation,TicketGroup,Title,Name_Length,Name_LengthGB,Name_Size
0,1,0,3,"Braund, Mr. Owen Harris",male,4,A/5 21171,7.2500,NaN,S,2,21171,A/5,BritishAgency,Mr,23,"(22.0, 28.0]",1.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,5,PC 17599,71.2833,C85,C,2,17599,PC,Premium,Mrs,51,"(28.0, 82.0]",2.0
2,3,1,3,"Heikkinen, Miss. Laina",female,4,STON/O2. 3101282,7.9250,NaN,S,1,3101282,STON/O2.,Southampton,Miss,22,"(11.999, 22.0]",0.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,5,113803,53.1000,C123,S,2,113803,Blank,Blank,Mrs,44,"(28.0, 82.0]",2.0
4,5,0,3,"Allen, Mr. William Henry",male,5,373450,8.0500,NaN,S,1,373450,Blank,Blank,Mr,24,"(22.0, 28.0]",1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,4,211536,13.0000,NaN,S,1,211536,Blank,Blank,Rev,21,"(11.999, 22.0]",0.0
887,888,1,1,"Graham, Miss. Margaret Edith",female,4,112053,30.0000,B42,S,1,112053,Blank,Blank,Miss,28,"(22.0, 28.0]",1.0
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,4,W./C. 6607,23.4500,NaN,S,4,6607,W./C.,BritishAgency,Miss,40,"(28.0, 82.0]",2.0
889,890,1,1,"Behr, Mr. Karl Howell",male,4,111369,30.0000,C148,C,1,111369,Blank,Blank,Mr,21,"(11.999, 22.0]",0.0


In [27]:
train_df['TicketNumberCounts'] = train_df.groupby('TicketNumber')['TicketNumber'].transform('count')
test_df['TicketNumberCounts'] = test_df.groupby('TicketNumber')['TicketNumber'].transform('count')
train_df.groupby(train_df["TicketNumberCounts"], as_index= False)["Survived"].mean()

,TicketNumberCounts,Survived
0,1,0.295956
1,2,0.569149
2,3,0.712121
3,4,0.500000
4,5,0.000000
5,6,0.000000
6,7,0.238095


In [28]:
train_df['Cabin_Assigned'] = train_df['Cabin'].apply(lambda x: 0 if x in ['U'] else 1)
test_df['Cabin_Assigned'] = test_df['Cabin'].apply(lambda x: 0 if x in ['U'] else 1)
train_df.groupby(['Cabin_Assigned'], as_index=False)['Survived'].agg(['count', 'mean'])

,Cabin_Assigned,count,mean
0,1,891,0.383838


In [29]:
# drop the cabin column as we already have the cabin assigned column
train_df.drop(columns= ["Cabin"], inplace=True)
test_df.drop(columns= ["Cabin"], inplace=True)

In [30]:
train_df["Fare"].describe()

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64

In [31]:
pd.qcut(train_df["Fare"], 5)

0        (-0.001, 7.854]
1      (39.688, 512.329]
2          (7.854, 10.5]
3      (39.688, 512.329]
4          (7.854, 10.5]
             ...        
886       (10.5, 21.679]
887     (21.679, 39.688]
888     (21.679, 39.688]
889     (21.679, 39.688]
890      (-0.001, 7.854]
Name: Fare, Length: 891, dtype: category
Categories (5, interval[float64, right]): [(-0.001, 7.854] < (7.854, 10.5] < (10.5, 21.679] < (21.679, 39.688] < (39.688, 512.329]]

In [32]:
train_df.loc[train_df['Fare'] <= 7.854, 'Fare'] = 0
train_df.loc[(train_df['Fare'] > 7.854) & (train_df['Fare'] <= 10.5), 'Fare'] = 1
train_df.loc[(train_df['Fare'] > 10.5) & (train_df['Fare'] <= 21.679), 'Fare'] = 2
train_df.loc[(train_df['Fare'] > 21.679) & (train_df['Fare'] <= 39.688), 'Fare'] = 3
train_df.loc[(train_df['Fare'] > 39.688) & (train_df['Fare'] <= 512.329), 'Fare'] = 4
train_df.loc[train_df['Fare'] > 512.329, 'Fare'] 

test_df.loc[test_df['Fare'] <= 7.854, 'Fare'] = 0
test_df.loc[(test_df['Fare'] > 7.854) & (test_df['Fare'] <= 10.5), 'Fare'] = 1
test_df.loc[(test_df['Fare'] > 10.5) & (test_df['Fare'] <= 21.679), 'Fare'] = 2
test_df.loc[(test_df['Fare'] > 21.679) & (test_df['Fare'] <= 39.688), 'Fare'] = 3
test_df.loc[(test_df['Fare'] > 39.688) & (test_df['Fare'] <= 512.329), 'Fare'] = 4
test_df.loc[test_df['Fare'] > 512.329, 'Fare'] 

343    512.3292
Name: Fare, dtype: float64

In [33]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   PassengerId         891 non-null    int64   
 1   Survived            891 non-null    int64   
 2   Pclass              891 non-null    int64   
 3   Name                891 non-null    object  
 4   Sex                 891 non-null    object  
 5   Age                 891 non-null    category
 6   Ticket              891 non-null    object  
 7   Fare                891 non-null    float64 
 8   Embarked            889 non-null    object  
 9   Family_Size         891 non-null    int64   
 10  TicketNumber        891 non-null    object  
 11  TicketLocation      891 non-null    object  
 12  TicketGroup         891 non-null    object  
 13  Title               891 non-null    object  
 14  Name_Length         891 non-null    int64   
 15  Name_LengthGB       891 non-null    cate

In [34]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   PassengerId         891 non-null    int64   
 1   Survived            891 non-null    int64   
 2   Pclass              891 non-null    int64   
 3   Name                891 non-null    object  
 4   Sex                 891 non-null    object  
 5   Age                 891 non-null    category
 6   Ticket              891 non-null    object  
 7   Fare                891 non-null    float64 
 8   Embarked            889 non-null    object  
 9   Family_Size         891 non-null    int64   
 10  TicketNumber        891 non-null    object  
 11  TicketLocation      891 non-null    object  
 12  TicketGroup         891 non-null    object  
 13  Title               891 non-null    object  
 14  Name_Length         891 non-null    int64   
 15  Name_LengthGB       891 non-null    cate

In [35]:
train_df.drop(columns = ["PassengerId", "Name", "Ticket", "TicketNumber"], inplace= True)
test_df.drop(columns = ["PassengerId", "Name", "Ticket", "TicketNumber"], inplace= True)

In [36]:
# TicketGroup is good enough, we don't need the ticketLocation col. Name size is good enough as we already classify names based on length, we don't need to know the name size bins as with namelengthGB
train_df.drop(columns= ["TicketLocation", "Name_LengthGB"], inplace= True)
test_df.drop(columns= ["TicketLocation", "Name_LengthGB"], inplace= True)

In [37]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   Survived            891 non-null    int64   
 1   Pclass              891 non-null    int64   
 2   Sex                 891 non-null    object  
 3   Age                 891 non-null    category
 4   Fare                891 non-null    float64 
 5   Embarked            889 non-null    object  
 6   Family_Size         891 non-null    int64   
 7   TicketGroup         891 non-null    object  
 8   Title               891 non-null    object  
 9   Name_Length         891 non-null    int64   
 10  Name_Size           891 non-null    float64 
 11  TicketNumberCounts  891 non-null    int64   
 12  Cabin_Assigned      891 non-null    int64   
dtypes: category(1), float64(2), int64(6), object(4)
memory usage: 84.9+ KB


In [38]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   Pclass              418 non-null    int64   
 1   Sex                 418 non-null    object  
 2   Age                 418 non-null    category
 3   Fare                417 non-null    float64 
 4   Embarked            418 non-null    object  
 5   Family_Size         418 non-null    int64   
 6   TicketGroup         418 non-null    object  
 7   Title               418 non-null    object  
 8   Name_Length         418 non-null    int64   
 9   Name_Size           418 non-null    float64 
 10  TicketNumberCounts  418 non-null    int64   
 11  Cabin_Assigned      418 non-null    int64   
dtypes: category(1), float64(2), int64(5), object(4)
memory usage: 36.8+ KB


In [39]:
scale_cols = [
    'Age',
    'Fare',
    'Name_Length'
]
ohe_cols = [
    "Sex", 
    "Embarked", 
    "TicketGroup", 
    "Title"
]
passthrough_cols = [
    'Pclass',
    'Family_Size',
    'Name_Size',
    'TicketNumberCounts',
    'Cabin_Assigned'
]

In [40]:
y = train_df["Survived"]
X = train_df.drop(columns= ["Survived"])
X_test = test_df
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify = y, random_state=21)

In [41]:
numeric_scaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [42]:
numeric_pass = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

In [43]:
categorical_ohe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [44]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num_scaled', numeric_scaled, scale_cols),
        ('num_pass', numeric_pass, passthrough_cols),
        ('cat_ohe', categorical_ohe, ohe_cols)
    ],
    remainder='drop'
)

In [45]:
rfc = RandomForestClassifier()

In [46]:
param_grid = {
    'n_estimators': [150, 200, 300, 500],
    'min_samples_split': [5, 10, 15],
    'max_depth': [10, 13, 15, 17, 20],
    'min_samples_leaf': [2, 4, 5, 6],
    'criterion': ['gini', 'entropy'],
}

In [47]:
CV_rfc = GridSearchCV(estimator=rfc, param_grid=param_grid, cv=StratifiedKFold(n_splits=5))

In [48]:
pipefinalrfc = make_pipeline(preprocessor, CV_rfc)
pipefinalrfc.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked',
                                                   'TicketGroup', 'Title'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=RandomForestClassifier(),
                              param_grid={'criterion': ['gini', 'entropy'],
                                          'max_depth': [10, 13, 15, 17, 20],
                                          'min_samples_leaf': [2, 4, 5, 6],
                                          'min_samples_split': [5, 10, 15],
                                          'n_estimators': [150, 200, 300,
                                                           500]}))])

In [49]:
print(CV_rfc.best_params_)
print(CV_rfc.best_score_)

{'criterion': 'gini', 'max_depth': 20, 'min_samples_leaf': 5, 'min_samples_split': 10, 'n_estimators': 150}
0.8370727863685609


In [50]:
dtc = DecisionTreeClassifier()

In [51]:
param_grid = {
    'min_samples_split': [5, 10, 15],
    'max_depth': [10, 20, 30],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy'],
}

In [52]:
CV_dtc = GridSearchCV(estimator=dtc, param_grid=param_grid, cv=StratifiedKFold(n_splits=5))

In [53]:
pipefinaldtc = make_pipeline(preprocessor, CV_dtc)
pipefinaldtc.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked',
                                                   'TicketGroup', 'Title'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=DecisionTreeClassifier(),
                              param_grid={'criterion': ['gini', 'entropy'],
                                          'max_depth': [10, 20, 30],
                                          'min_samples_leaf': [1, 2, 4],
                                          'min_samples_split': [5, 10, 15]}))])

In [54]:
print(CV_dtc.best_params_)
print(CV_dtc.best_score_)

{'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 5}
0.8202107751403526


In [55]:
knn = KNeighborsClassifier()

In [56]:
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'p': [1,2],
}

In [57]:
CV_knn = GridSearchCV(estimator=knn, param_grid=param_grid, cv=StratifiedKFold(n_splits=5))

In [58]:
pipefinalknn = make_pipeline(preprocessor, CV_knn)
pipefinalknn.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked',
                                                   'TicketGroup', 'Title'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=KNeighborsClassifier(),
                              param_grid={'algorithm': ['auto', 'ball_tree',
                                                        'kd_tree', 'brute'],
                                          'n_neighbors': [3, 5, 7, 9, 11],
                                          'p': [1, 2],
                                          'weights': ['uniform',
                                                      'distance']}))])

In [59]:
print(CV_knn.best_params_)
print(CV_knn.best_score_)

{'algorithm': 'auto', 'n_neighbors': 7, 'p': 1, 'weights': 'distance'}
0.8202600216684722


In [60]:
svc = SVC(probability=True)

In [61]:
param_grid = {
    'C': [100,10, 1.0, 0.1, 0.001, 0.001],
    'kernel':['linear', 'poly', 'rbf', 'sigmoid'],
}

In [62]:
CV_svc = GridSearchCV(estimator=svc, param_grid=param_grid, cv=StratifiedKFold(n_splits=5))

In [63]:
pipefinalsvc = make_pipeline(preprocessor, CV_svc)
pipefinalsvc.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked',
                                                   'TicketGroup', 'Title'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=SVC(probability=True),
                              param_grid={'C': [100, 10, 1.0, 0.1, 0.001,
                                                0.001],
                                          'kernel': ['linear', 'poly', 'rbf',
                                                     'sigmoid']}))])

In [64]:
print(CV_svc.best_params_)
print(CV_svc.best_score_)

{'C': 100, 'kernel': 'linear'}
0.8384812370727864


In [65]:
lr = LogisticRegression(max_iter=1000)

In [66]:
param_grid = {
    'C': [100,10, 1.0, 0.1, 0.001, 0.001],
}

In [67]:
CV_lr = GridSearchCV(estimator=lr, param_grid=param_grid, cv=StratifiedKFold(n_splits=5))

In [68]:
pipefinallr= make_pipeline(preprocessor, CV_lr)
pipefinallr.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked',
                                                   'TicketGroup', 'Title'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=LogisticRegression(max_iter=1000),
                              param_grid={'C': [100, 10, 1.0, 0.1, 0.001,
                                                0.001]}))])

In [69]:
print(CV_lr.best_params_)
print(CV_lr.best_score_)

{'C': 100}
0.8357135821924555


In [70]:
gnb = GaussianNB()

In [71]:
param_grid = {
    'var_smoothing': [0.00000001, 0.000000001, 0.00000001],
}

In [72]:
CV_gnb = GridSearchCV(estimator=gnb, param_grid=param_grid, cv=StratifiedKFold(n_splits=5))

In [73]:
pipefinalgnb= make_pipeline(preprocessor, CV_gnb)
pipefinalgnb.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                 ('cat_ohe',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked',
                                                   'TicketGroup', 'Title'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=GaussianNB(),
                              param_grid={'var_smoothing': [1e-08, 1e-09,
                                                            1e-08]}))])

In [74]:
print(CV_gnb.best_params_)
print(CV_gnb.best_score_)

{'var_smoothing': 1e-09}
0.4466364621294199


In [75]:
xg = XGBClassifier()

In [76]:
param_grid = {
     'booster': ['gbtree', 'gblinear','dart'],
}

In [77]:
CV_xg = GridSearchCV(estimator=xg, param_grid=param_grid, cv=StratifiedKFold(n_splits=5))

In [78]:
pipefinalxg= make_pipeline(preprocessor, CV_xg)
pipefinalxg.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                      interaction_constraints=None,
                                                      learning_rate=None,
                                                      max_bin=None,
                                                      max_cat_threshold=None,
                                                      max_cat_to_onehot=None,
                                                      max_delta_step=None,
                                                      max_depth=None,
                                                      max_leaves=None,
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=None,
                                                      num_parallel_tree=None, ...),
                              param_grid={'booster': ['gbtree', 'gblinear',
                                                      'dart']}))])

In [79]:
print(CV_xg.best_params_)
print(CV_xg.best_score_)

{'booster': 'gblinear'}
0.8314882300797795


In [80]:
abc = AdaBoostClassifier()

In [81]:
dtc_2 = DecisionTreeClassifier(criterion = 'entropy', max_depth=10,min_samples_leaf=4, min_samples_split=10)  
svc_2 = SVC(probability=True, C=10, kernel='rbf') 
lr_2 = LogisticRegression(C=0.1) 
lr_3 = LogisticRegression(C=0.2) 
lr_4 = LogisticRegression(C=0.05) 

In [82]:
param_grid = {
    'estimator': [dtc_2, svc_2, lr_2], 
    'n_estimators':  [5, 10, 25, 50, 100],
    'learning_rate': [(0.97 + x / 100) for x in range(1, 7)]  
}

In [83]:
CV_abc = GridSearchCV(estimator=abc, param_grid=param_grid, cv=StratifiedKFold(n_splits=5))

In [84]:
pipefinalabc= make_pipeline(preprocessor, CV_abc)
pipefinalabc.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=AdaBoostClassifier(),
                              param_grid={'estimator': [DecisionTreeClassifier(criterion='entropy',
                                                                               max_depth=10,
                                                                               min_samples_leaf=4,
                                                                               min_samples_split=10),
                                                        SVC(C=10,
                                                            probability=True),
                                                        LogisticRegression(C=0.1)],
                                          'learning_rate': [0.98, 0.99, 1.0,
                                                            1.01, 1.02, 1.03],
                                          'n_estimators': [5, 10, 25, 50,
                                                           100]}))])

In [85]:
print(CV_abc.best_params_)
print(CV_abc.best_score_)

{'estimator': DecisionTreeClassifier(criterion='entropy', max_depth=10, min_samples_leaf=4,
                       min_samples_split=10), 'learning_rate': 1.03, 'n_estimators': 100}
0.8147345612134345


In [86]:
etc = ExtraTreesClassifier()

In [87]:
param_grid = {
              "max_features": [1, 3, 10],
              "min_samples_split": [2, 3, 10],
              "min_samples_leaf": [1, 3, 10],
              "n_estimators" :[100,300],
}

In [88]:
CV_etc = GridSearchCV(estimator=etc, param_grid=param_grid, cv=StratifiedKFold(n_splits=5))

In [89]:
pipefinaletc= make_pipeline(preprocessor, CV_etc)
pipefinaletc.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked',
                                                   'TicketGroup', 'Title'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=ExtraTreesClassifier(),
                              param_grid={'max_features': [1, 3, 10],
                                          'min_samples_leaf': [1, 3, 10],
                                          'min_samples_split': [2, 3, 10],
                                          'n_estimators': [100, 300]}))])

In [90]:
print(CV_etc.best_params_)
print(CV_etc.best_score_)

{'max_features': 10, 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 300}
0.8328966807840048


In [91]:
GBC = GradientBoostingClassifier()

In [92]:
param_grid = {
              'n_estimators' : [300, 400, 500],
              'learning_rate': [ 0.1, 0.3, 0.6, 1.0],
              'max_depth': [8, 10, 12],
              'min_samples_leaf': [50, 100, 120, 150],
              'max_features': [0.1, 0.3, 0.5] 
              }


In [93]:
CV_gbc = GridSearchCV(estimator=GBC, param_grid=param_grid, cv=StratifiedKFold(n_splits=5))

In [94]:
pipefinalgbc= make_pipeline(preprocessor, CV_gbc)
pipefinalgbc.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked',
                                                   'TicketGroup', 'Title'])])),
                ('gridsearchcv',
                 GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                              estimator=GradientBoostingClassifier(),
                              param_grid={'learning_rate': [0.1, 0.3, 0.6, 1.0],
                                          'max_depth': [8, 10, 12],
                                          'max_features': [0.1, 0.3, 0.5],
                                          'min_samples_leaf': [50, 100, 120,
                                                               150],
                                          'n_estimators': [300, 400, 500]}))])

In [95]:
print(CV_gbc.best_params_)
print(CV_gbc.best_score_)

{'learning_rate': 0.6, 'max_depth': 8, 'max_features': 0.1, 'min_samples_leaf': 50, 'n_estimators': 300}
0.8202600216684723


In [96]:
vc1 = VotingClassifier([('gbc', CV_gbc.best_estimator_),
                        ('etc', CV_etc.best_estimator_),
                          ('nb', CV_gnb.best_estimator_)
                         ], voting='hard', weights=[1,2,3] )

In [97]:
vc2 = VotingClassifier([('abc', CV_abc.best_estimator_),
                        ('etc', CV_etc.best_estimator_),
                          ('nb', CV_gnb.best_estimator_)
                         ], voting='hard', weights=[1,2,3] )
#1,2,3 is the best performing one

In [98]:
pipefinalcv1 = make_pipeline(preprocessor, vc1)

In [99]:
pipefinalcv2 = make_pipeline(preprocessor, vc2)

In [100]:
pipefinalcv1.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked',
                                                   'TicketGroup', 'Title'])])),
                ('votingclassifier',
                 VotingClassifier(estimators=[('gbc',
                                               GradientBoostingClassifier(learning_rate=0.6,
                                                                          max_depth=8,
                                                                          max_features=0.1,
                                                                          min_samples_leaf=50,
                                                                          n_estimators=300)),
                                              ('etc',
                                               ExtraTreesClassifier(max_features=10,
                                                                    min_samples_split=3,
                                                                    n_estimators=300)),
                                              ('nb', GaussianNB())],
                                  weights=[1, 2, 3]))])

In [101]:
pipefinalcv2.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num_scaled',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'Name_Length']),
                                                 ('num_pass',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Family_Size',
                                                   'Name_Size',
                                                   'TicketNumberCounts',
                                                   'Cabin_Assigne...
                                                   'TicketGroup', 'Title'])])),
                ('votingclassifier',
                 VotingClassifier(estimators=[('abc',
                                               AdaBoostClassifier(estimator=DecisionTreeClassifier(criterion='entropy',
                                                                                                   max_depth=10,
                                                                                                   min_samples_leaf=4,
                                                                                                   min_samples_split=10),
                                                                  learning_rate=1.03,
                                                                  n_estimators=100)),
                                              ('etc',
                                               ExtraTreesClassifier(max_features=10,
                                                                    min_samples_split=3,
                                                                    n_estimators=300)),
                                              ('nb', GaussianNB())],
                                  weights=[1, 2, 3]))])

In [102]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   Pclass              418 non-null    int64   
 1   Sex                 418 non-null    object  
 2   Age                 418 non-null    category
 3   Fare                417 non-null    float64 
 4   Embarked            418 non-null    object  
 5   Family_Size         418 non-null    int64   
 6   TicketGroup         418 non-null    object  
 7   Title               418 non-null    object  
 8   Name_Length         418 non-null    int64   
 9   Name_Size           418 non-null    float64 
 10  TicketNumberCounts  418 non-null    int64   
 11  Cabin_Assigned      418 non-null    int64   
dtypes: category(1), float64(2), int64(5), object(4)
memory usage: 36.8+ KB


In [103]:
Y_pred = pipefinalrfc.predict(X_test)
Y_pred2 = pipefinaldtc.predict(X_test)
Y_pred3 = pipefinalknn.predict(X_test)
Y_pred4 = pipefinalsvc.predict(X_test)
Y_pred5 = pipefinallr.predict(X_test)
Y_pred6 = pipefinalgnb.predict(X_test)
Y_pred7 = pipefinalxg.predict(X_test)
Y_pred8 = pipefinalabc.predict(X_test)
Y_pred9 = pipefinaletc.predict(X_test)
Y_pred10 = pipefinalgbc.predict(X_test)
Y_pred11 = pipefinalcv1.predict(X_test)
Y_pred12 = pipefinalcv2.predict(X_test)

In [104]:
submission = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': Y_pred
})

submission2 = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': Y_pred2
})

submission3 = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': Y_pred3
})

submission4 = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': Y_pred4
})

submission5 = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': Y_pred5
})

submission6 = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': Y_pred6
})

submission7 = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': Y_pred7
})

submission8 = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': Y_pred8
})

submission9 = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': Y_pred9
})

submission10 = pd.DataFrame({
    'PassengerId': submission_df['PassengerId'],
    'Survived': Y_pred10
})

submission11 = pd.DataFrame({
        "PassengerId": submission_df["PassengerId"],
        "Survived": Y_pred11
})

submission12 = pd.DataFrame({
        "PassengerId": submission_df["PassengerId"],
        "Survived": Y_pred12
})

In [105]:
submission.to_csv('/kaggle/working/submission182_1.csv', index=False)
submission2.to_csv('/kaggle/working/submission182_2.csv', index=False)
submission3.to_csv('/kaggle/working/submission182_3.csv', index=False)
submission4.to_csv('/kaggle/working/submission182_4.csv', index=False)
submission5.to_csv('/kaggle/working/submission182_5.csv', index=False)
submission6.to_csv('/kaggle/working/submission182_6.csv', index=False)
submission7.to_csv('/kaggle/working/submission182_7.csv', index=False)
submission8.to_csv('/kaggle/working/submission182_8.csv', index=False)
submission9.to_csv('/kaggle/working/submission182_9.csv', index=False)
submission10.to_csv('/kaggle/working/submission182_10.csv', index=False)
submission11.to_csv('/kaggle/working/submission182_11.csv', index=False)
submission12.to_csv('/kaggle/working/submission182_12.csv', index=False)